# Studio di MERGE
basato su cleaning 4
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from data_model.manage_excel_support_file import *
from data_model.MergerTools import *
import pandas as pd
import os

client = DatalakeClient()
mergeTools = MergerTools()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake


**Mixed info**\
'ADNIMERGE', \
'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', --> have CSF

**Single Cofactor**\
'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES'

**Volumes**\
'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS',\
'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T'  --> just partial immages segmentation

**CSF**\
'UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', 'EUROIMMUN', 'FUJIREBIOABETA', 'SALADAX_BIOMEDICAL', 'MESOSCALE', 'UPENNBIOMK_MASTER', 'UPENN_2DUPLC_CRM', 

In [2]:
file_codes =['ADNIMERGE', 'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', 'ADAS', 'FAQ', 'CDR', 'MMSE', 'MOCA'] #
search = client.query_files(
    query={'custom.level' : 'cleaned_04', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(len(zip_files))

8


In [3]:
##### MODIFICARE 
category = 'scale'

In [4]:
dfs = {}
df_names = {}
df_code = []
idx = 0
for file_name, df_raw in zip_files.items():
    print('\n### ', file_name)
    df_copy = df_raw.copy(deep=True)
    len_pre = len(df_copy.columns)
    row_pre = len(df_copy)
    # get the sub_df focusing on the category chosen
    df_copy = mergeTools.filter_df_category(df_copy, category)
    len_post = len(df_copy.columns)
    row_post = len(df_copy)
    if df_copy.empty:
        print(file_name, f'------> non ha colonne dellca categoria {category}')
        continue
    # get the common columns among all the dfs
    if idx == 0:
        dfs_columns = set(df_copy.columns)
    else: 
        dfs_columns &= set(df_copy.columns)
    # Ensure EXAMDATE in date format and correct order of the dfs
    df_copy['EXAMDATE'] = pd.to_datetime(df_copy['EXAMDATE'])
    df_copy = df_copy.sort_values(by=['RID', 'EXAMDATE']).reset_index(drop=True)
    if 'FSVERSION' in df_copy.columns:
        df_copy['FSVERSION'] = df_copy['FSVERSION'].astype(str)
    # aggiornamento liste e dizionari
    dfs[f"df_{idx}"] = df_copy  
    df_names[f"df_{idx}"] = file_name 
    df_code.append(f"df_{idx}")
    # definizione variabile df
    globals()[f"df_{idx}"] = df_copy
    print(idx, '--->', file_name, '\t\t\t### ', len_post, '/', len_pre, '\n\t\t\t\t\t\t rows: ', row_post, '/', row_pre)
    idx += 1

time_buffer = pd.Timedelta(days=80)


###  ADNIMERGE_25Jul2025_04.csv
0 ---> ADNIMERGE_25Jul2025_04.csv 			###  13 / 42 
						 rows:  11452 / 11458

###  MMSE_25Jul2025_04.csv
1 ---> MMSE_25Jul2025_04.csv 			###  7 / 7 
						 rows:  14335 / 14335

###  ADAS_28Oct2025_04.csv
2 ---> ADAS_28Oct2025_04.csv 			###  8 / 8 
						 rows:  12648 / 12648

###  FAQ_28Oct2025_04.csv
3 ---> FAQ_28Oct2025_04.csv 			###  7 / 7 
						 rows:  13007 / 13007

###  CDR_28Oct2025_04.csv
4 ---> CDR_28Oct2025_04.csv 			###  7 / 8 
						 rows:  14347 / 14350

###  MOCA_28Oct2025_04.csv
5 ---> MOCA_28Oct2025_04.csv 			###  7 / 7 
						 rows:  3689 / 3689

###  ADSP_PHC_BIOMARKER_25Jul2025_04.csv
ADSP_PHC_BIOMARKER_25Jul2025_04.csv ------> non ha colonne dellca categoria scale

###  ADNI-DIAN_Comparison_Study_Data_Subset_05_23_22_23Oct2025_04.csv
6 ---> ADNI-DIAN_Comparison_Study_Data_Subset_05_23_22_23Oct2025_04.csv 			###  9 / 41 
						 rows:  3634 / 3673


In [5]:
dfs_columns

{'COHORT', 'EXAMDATE', 'RID', 'VISCODE', 'VISIT_MONTH', 'update_stamp'}

In [ ]:
x = 0
for df_x in dfs.values():
    print(x, df_x['update_stamp'].unique())
    x += 1

# Confronto stessi RID  ==> RID - EXAMDATE identici tra file

In [7]:

subj_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID'])
print("righe con stessi ####### RID:")
display(subj_matrix)
        
subj_date_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE'], time_buffer=time_buffer)
print("righe con stessi ####### RID-EXAMDATE: --> time_buffer=", time_buffer)
display(subj_date_matrix)

if set(['FSVERSION', 'IMAGEUID']).issubset(set(dfs_columns)):
    subj_viscode_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE', 'FSVERSION','IMAGEUID'])
    print("righe con stessi ####### RID-EXAMDATE-FSVERSION-IMAGEUID: --> time_buffer=", time_buffer)
    display(subj_viscode_matrix)

if set('METHOD').issubset(set(dfs_columns)):
    subj_viscode_matrix = mergeTools.matrix_match(dfs, df_names, df_code, columns_list=['RID', 'EXAMDATE', 'METHOD'])
    print("righe con stessi ####### RID-EXAMDATE-METHOD: --> time_buffer=", time_buffer)
    display(subj_viscode_matrix)

righe con stessi ####### RID:


,df_0,df_1,df_2,df_3,df_4,df_5,df_6
df_0,2409,,,,,,
df_1,2409,4700,,,,,
df_2,2406,2996,3012,,,,
df_3,2400,2982,2985,2999,,,
df_4,2409,4296,3011,2998,4351,,
df_5,1074,1654,1664,1648,1669,1670,
df_6,571,571,571,570,571,103,571


righe con stessi ####### RID-EXAMDATE: --> time_buffer= 80 days 00:00:00


,df_0,df_1,df_2,df_3,df_4,df_5,df_6
df_0,11452,,,,,,
df_1,11171,14332,,,,,
df_2,11327,12123,12645,,,,
df_3,11239,11954,12341,13004,,,
df_4,11045,13656,12012,12444,14345,,
df_5,2392,3286,3626,3487,3285,3688,
df_6,3153,3104,3134,3186,3146,0,3213


# Inizio Merge
## Definizione di df_base e Gerarchia di DF da mergiare
Scegliere il file con numero maggiore di soggetti-visite e che ha più elementi con altri df.\
Quindi scegliere con che ordine unire gli altri df, suggerimento da quelli con nessuna/pochissime righe RID-EXAMDATE in comune con gli altri df, e quindi quelli con molte righe in comune a partire da quello con più righe in comune sia con df_base che con gli altri e quindi a seguire. Ma di persè il metodo è arbitrario quindi si può fare come si vuole.

In [8]:
############   MODIFICARE
base = 'df_0'
df_base = dfs[base].copy(deep=True)                           #sembra un errore ma questi df sono definiti
idx_add = [ 'df_6', 'df_1', 'df_2', 'df_3', 'df_4', 'df_5']
merge_contains = [base]
sub_with_match = set()
time_buffer = pd.Timedelta(days=80)
i = 0

## Approfondimento RID-EXAMDATE
1. vedo quante righe ci sono con match ESATTO e quante con TIME BUFFER.

In [9]:
df_add = dfs[idx_add[i]].copy(deep=True)
print(idx_add[i])

df_6


In [10]:
exact_matches, buffer_matches = mergeTools.find_visit_matches(df_base, df_add, buffer_days=time_buffer)
exact_index1, exact_index2 = mergeTools.list_index_visit_matches(exact_matches)
buff_index1, buff_index2 = mergeTools.list_index_visit_matches(buffer_matches)


print('Exact matches: \t', len(exact_index1))
print('Buffered matches: \t', len(buff_index1))

if len(buff_index1) != len(buff_index2):
    print('\n ------>>> ATTENZIONE: indici match con buffer SPAIATI')

if len(buff_index1) != len(buff_index2):
    print('\n ------>>> ATTENZIONE: indici match SPAIATI')

Exact matches: 	 3572
Buffered matches: 	 2


In [11]:
columns_in_common = list(df_base.columns.intersection(df_add.columns))
columns_only_base = list(df_base.columns.difference(df_add.columns))
columns_only_add = list(df_add.columns.difference(df_base.columns))

all_index1 = exact_index1.union(buff_index1)
all_index2 = exact_index2.union(buff_index2)

# studio le colonne in comune e non ai due df
print('Le colonne in comune sono:\n', columns_in_common)
print('\n\nLe colonne solo in df_base sono: \n', columns_only_base)
print('\n\nLe colonne solo in df_add sono: \n', columns_only_add)
print('__________________________________________________________________\n\n')

diff_date = False
# verifico ci siano righe matchate con BUFFER
if len(buff_index1) == len(buff_index2) and len(buff_index1) > 0:    
    diff_date = True
    # verifico se le righe matchate sono TUTTE matchate con BUFFER
    if all(all_index1) == len(all_index2) and all_index1 == buff_index1 and all_index2 == buff_index2:
        print(f'all maches have a buffer, tot: {len(all_index1)} matches\n\n====================================> Renamen\'EXAMDATE\' column\n')
    else:
        print(f'There are {len(buff_index1)} match with buffer\nThere are {len(all_index1)-len(buff_index1)} match exact\nOver {len(all_index1)} total matches\n\n====================================> SHOULD \'EXAMDATE\' column be renamed?\n')
    

# ci sono solo match ESATTI
elif len(buff_index1) == len(buff_index2) and len(buff_index1) == 0 and len(all_index1) > 0:
    print('JUST exact matches')
elif len(buff_index1) == len(buff_index2) and len(buff_index1) == 0 and len(all_index1) == 0:
    print('NO matches')


Le colonne in comune sono:
 ['RID', 'COHORT', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'CDRSB', 'MMSE', 'update_stamp']


Le colonne solo in df_base sono: 
 ['ADAS11', 'ADAS13', 'FAQ', 'MOCA', 'RAVLT_immediate']


Le colonne solo in df_add sono: 
 ['CDRGLOB']
__________________________________________________________________


There are 2 match with buffer
There are 3572 match exact
Over 3574 total matches

====================================> SHOULD 'EXAMDATE' column be renamed?



In [12]:
if diff_date:
    print('df_add --> ', df_names[idx_add[i]], '\ndf_base --> ', [df_names[x] for x in merge_contains])
    col_list = list(df_base.columns) + [c for c in df_add.columns if c not in df_base.columns]
    
    temp_merge = mergeTools.create_temp_merge(df_base, df_add, buff_index1, buff_index2, col_list=col_list)
    diff = temp_merge['EXAMDATE_1']-temp_merge['EXAMDATE_2']
    display(diff[diff != pd.Timedelta(days=0)])
    print(len(diff[diff != pd.Timedelta(days=0)]))
    display(temp_merge.loc[diff[diff != pd.Timedelta(days=0)].index])
    

df_add -->  ADNI-DIAN_Comparison_Study_Data_Subset_05_23_22_23Oct2025_04.csv 
df_base -->  ['ADNIMERGE_25Jul2025_04.csv']


0   -2 days
1   21 days
dtype: timedelta64[ns]

2


,RID,COHORT_1,COHORT_2,VISCODE_1,VISCODE_2,VISIT_MONTH_1,VISIT_MONTH_2,EXAMDATE_1,EXAMDATE_2,CDRSB_1,...,RAVLT_immediate_1,RAVLT_immediate_2,FAQ_1,FAQ_2,MOCA_1,MOCA_2,update_stamp_1,update_stamp_2,CDRGLOB_1,CDRGLOB_2
0,4294,ADNI2,nv,m48,nv,48,48,2015-12-16,2015-12-18,3.5,...,30.0,None,7.0,None,21.0,None,2023-07-07 04:59:56.0,2022-05-25 15:48:06.0,None,NaN
1,4499,ADNI2,ADNI2,m12,m12,12,12,2013-04-09,2013-03-19,1.0,...,27.0,None,0.0,None,28.0,None,2023-07-07 04:59:58.0,2022-05-25 15:48:07.0,None,0.5


In [ ]:
# best practice automatica
modify_examdate = False #True   #False
index_modified = []
if modify_examdate:
    for x1, x2 in zip(buff_index1, buff_index2):
        if df_base.loc[x1, 'VISCODE'] == df_add.loc[x2, 'VISCODE'] or df_base.loc[x1, 'EXAMDATE'] - df_add.loc[x2, 'EXAMDATE'] < pd.Timedelta(days=30):
            df_add.loc[x2, 'EXAMDATE'] = df_base.loc[x1, 'EXAMDATE']
            index_modified.append([x1, x2])

print(len(index_modified))

2


In [ ]:
modify_examdate = False #True   #False
if modify_examdate:
    df_add.loc[buff_index2, 'EXAMDATE'] = df_base.loc[buff_index1, 'EXAMDATE'].values
    display(df_add.loc[buff_index2]['EXAMDATE'])

## Studio Colonne in comune per righe che matchano
1) Se ci sono righe che machano tra i due df allora identifico altre colonne in comune ai due df.

2) Faccio merge tra i due df escludendo i soggetti che hanno visite metchate tra i 2 df (righe). --> merge_base solo aggiunta di soggetti nuovi.

3) Quindi se ci sono colonne in comune e righe che matchano faccio merge soggetto per soggetto (tra i soggetti con  visite in entrambi i df).\
Aggiungo qusti merge di singoli soggetti al resto del merge_base.


Così ottengo Merge finale.

In [ ]:
df_merged = mergeTools.get_merged_df(df_add, df_base, category=category)

In [ ]:
df_base = df_merged.copy(deep=True)
merge_contains.append(idx_add[i])
#prepare for next merge
i += 1
print(f'il merg contine i seguenti df: {merge_contains}')
if i <= len(idx_add)-1:
    print(f'il prossimo df da unire è: {idx_add[i]}\n\n ===> torna al capitolo: "Approfondimento RID-EXAMDATE"')
else:
    print('FINISHED MERGE!!!!!')

# SALVAREEEE

In [ ]:
new_file_name = category + '_merged.csv'
file_code = 'VOLMERGE'

# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=df_merged,
    object_name=new_file_name,
    prefix='cleaned/merged/category',
    metadata={
        'level': 'merged',
        'file_code': file_code,
        'source': 'ADNI'
    }
)